# Fine-tune DistilBERT for Seven-Emotion YouTube Comment Analysis

This Colab notebook trains the project emotion model in two stages:

1. Fine-tune `distilbert-base-uncased` on a balanced seven-class GoEmotions dataset.
2. Continue training on the expanded YouTube-domain adaptation dataset with 2,962 assistant-assisted comments from 30 videos.

Target labels: `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, and `surprise`.

Recommended Colab runtime: **T4 GPU**.


## 1. Install dependencies

Run this cell first, then restart the runtime only if Colab asks for it.


In [ ]:
!pip install -q "transformers>=4.46" datasets accelerate evaluate scikit-learn pandas pyarrow huggingface_hub


## 2. Import packages and define labels


In [ ]:
import inspect
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline

SEED = 5240
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABEL_TO_ID = {label: index for index, label in enumerate(TARGET_LABELS)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}

print("Labels:", TARGET_LABELS)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Helper functions for compatible training

Different `transformers` versions use slightly different argument names. These helpers make the notebook safer on Colab.


In [ ]:
def make_training_args(**kwargs):
    signature = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in signature:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    else:
        kwargs["evaluation_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    return TrainingArguments(**kwargs)


def make_trainer(model, args, train_dataset, eval_dataset, tokenizer, compute_metrics):
    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "compute_metrics": compute_metrics,
    }
    signature = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in signature:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in signature:
        trainer_kwargs["tokenizer"] = tokenizer
    return Trainer(**trainer_kwargs)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
    }


## 4. Load and prepare GoEmotions dataset

The raw GoEmotions dataset is multi-label. We keep clean single-label examples where exactly one label is active and the active label is one of the seven target emotions. Then we balance each split by downsampling to the smallest class.


In [ ]:
PARQUET_URLS = {
    "train": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}


def filter_single_target_emotions(df):
    all_label_columns = [column for column in df.columns if column != "text"]
    single_label_df = df[df[all_label_columns].sum(axis=1) == 1].copy()
    target_df = single_label_df[single_label_df[TARGET_LABELS].sum(axis=1) == 1].copy()
    target_df["label_name"] = target_df[TARGET_LABELS].idxmax(axis=1)
    target_df["label"] = target_df["label_name"].map(LABEL_TO_ID).astype(int)
    return target_df[["text", "label_name", "label"]].reset_index(drop=True)


def balance_by_label(df, random_state=SEED):
    min_count = df["label_name"].value_counts().min()
    parts = []
    for label_name in sorted(df["label_name"].unique()):
        parts.append(df[df["label_name"] == label_name].sample(n=min_count, random_state=random_state))
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)


prepared_frames = {}
for split, url in PARQUET_URLS.items():
    raw_df = pd.read_parquet(url)
    filtered_df = filter_single_target_emotions(raw_df)
    prepared_frames[split] = balance_by_label(filtered_df)
    print(split, prepared_frames[split].shape)
    print(prepared_frames[split]["label_name"].value_counts().sort_index())


## 5. Convert GoEmotions data to Hugging Face Datasets


In [ ]:
go_dataset = DatasetDict({
    split: Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    for split, df in prepared_frames.items()
})

go_dataset


## 6. Tokenize text


In [ ]:
BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

go_tokenized_dataset = go_dataset.map(tokenize_batch, batched=True)
go_tokenized_dataset = go_tokenized_dataset.remove_columns(["text"])
go_tokenized_dataset.set_format("torch")

go_tokenized_dataset


## 7. Load base model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(TARGET_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)


## 8. Stage 1 training: GoEmotions seven-class model


In [ ]:
training_args = make_training_args(
    output_dir="./youtube-emotion-distilbert-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = make_trainer(
    model=model,
    args=training_args,
    train_dataset=go_tokenized_dataset["train"],
    eval_dataset=go_tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

stage1_start = time.time()
trainer.train()
print("Stage 1 training seconds:", round(time.time() - stage1_start, 2))


## 9. Evaluate Stage 1 model


In [ ]:
stage1_validation_metrics = trainer.evaluate(go_tokenized_dataset["validation"])
stage1_test_metrics = trainer.evaluate(go_tokenized_dataset["test"])

print("Stage 1 validation metrics:", stage1_validation_metrics)
print("Stage 1 test metrics:", stage1_test_metrics)


## 10. Save and smoke-test Stage 1 model


In [ ]:
MODEL_OUTPUT_DIR = "./youtube-emotion-distilbert"
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Saved Stage 1 model to {MODEL_OUTPUT_DIR}")

samples = [
    "I love this video so much!",
    "This campaign makes me angry.",
    "I am shocked by this announcement.",
    "This is just a normal update.",
]

emotion_pipeline = pipeline("text-classification", model=MODEL_OUTPUT_DIR, tokenizer=MODEL_OUTPUT_DIR)
emotion_pipeline(samples, truncation=True, return_token_type_ids=False)


## 11. Load expanded YouTube-domain adaptation dataset

This dataset is stored in the GitHub repository and currently contains:

- Train: 2,370 comments
- Validation: 592 comments
- Total: 2,962 comments from 30 YouTube videos

The labels are assistant-assisted and are used only for domain adaptation. The independent app benchmark remains separate.


In [ ]:
YOUTUBE_DATA_BASE = "https://raw.githubusercontent.com/chasezhang1999/youtube-emotion-analyzer/main/data/youtube_domain_7class_assistant"

youtube_train_df = pd.read_csv(f"{YOUTUBE_DATA_BASE}/train.csv")
youtube_val_df = pd.read_csv(f"{YOUTUBE_DATA_BASE}/validation.csv")

for df in [youtube_train_df, youtube_val_df]:
    df["text"] = df["text"].astype(str)
    df["label_name"] = df["label"].astype(str)
    df["label"] = df["label_id"].astype(int)

print("YouTube-domain train:", youtube_train_df.shape)
print(youtube_train_df["label_name"].value_counts().sort_index())
print("YouTube-domain validation:", youtube_val_df.shape)
print(youtube_val_df["label_name"].value_counts().sort_index())


## 12. Convert and tokenize YouTube-domain data


In [ ]:
domain_dataset = DatasetDict({
    "train": Dataset.from_pandas(youtube_train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(youtube_val_df[["text", "label"]], preserve_index=False),
})

domain_tokenized_dataset = domain_dataset.map(tokenize_batch, batched=True)
domain_tokenized_dataset = domain_tokenized_dataset.remove_columns(["text"])
domain_tokenized_dataset.set_format("torch")

domain_tokenized_dataset


## 13. Stage 2 training: YouTube-domain adaptation

Because the YouTube-domain dataset is naturally imbalanced, this stage uses `macro_f1` for best-model selection instead of accuracy. This makes the validation metric more sensitive to minority classes such as fear, sadness, disgust, and surprise.


In [ ]:
domain_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_OUTPUT_DIR,
    num_labels=len(TARGET_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

domain_training_args = make_training_args(
    output_dir="./youtube-emotion-distilbert-domain-results",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=25,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

domain_trainer = make_trainer(
    model=domain_model,
    args=domain_training_args,
    train_dataset=domain_tokenized_dataset["train"],
    eval_dataset=domain_tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

stage2_start = time.time()
domain_trainer.train()
print("Stage 2 training seconds:", round(time.time() - stage2_start, 2))


## 14. Evaluate and save domain-adapted model


In [ ]:
domain_validation_metrics = domain_trainer.evaluate(domain_tokenized_dataset["validation"])
print("YouTube-domain validation metrics:", domain_validation_metrics)

DOMAIN_MODEL_OUTPUT_DIR = "./youtube-emotion-distilbert-domain-adapted"
domain_trainer.save_model(DOMAIN_MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(DOMAIN_MODEL_OUTPUT_DIR)
print(f"Saved domain-adapted model to {DOMAIN_MODEL_OUTPUT_DIR}")

domain_pipeline = pipeline("text-classification", model=DOMAIN_MODEL_OUTPUT_DIR, tokenizer=DOMAIN_MODEL_OUTPUT_DIR)
domain_pipeline(samples, truncation=True, return_token_type_ids=False)


## 15. Compare Stage 1 and Stage 2 sample predictions


In [ ]:
comparison_samples = [
    "This launch is amazing and I want to buy it now!",
    "The brand response is terrible and people are angry.",
    "This safety ad is scary but important.",
    "I did not expect that ending at all.",
    "Just here to check the product details.",
]

stage1_pipeline = pipeline("text-classification", model=MODEL_OUTPUT_DIR, tokenizer=MODEL_OUTPUT_DIR)
stage2_pipeline = pipeline("text-classification", model=DOMAIN_MODEL_OUTPUT_DIR, tokenizer=DOMAIN_MODEL_OUTPUT_DIR)

comparison_rows = []
for text in comparison_samples:
    stage1_pred = stage1_pipeline(text, truncation=True, return_token_type_ids=False)[0]
    stage2_pred = stage2_pipeline(text, truncation=True, return_token_type_ids=False)[0]
    comparison_rows.append({
        "text": text,
        "stage1_label": stage1_pred["label"],
        "stage1_score": round(stage1_pred["score"], 4),
        "domain_label": stage2_pred["label"],
        "domain_score": round(stage2_pred["score"], 4),
    })

pd.DataFrame(comparison_rows)


## 16. Upload models to Hugging Face

Before running this section:

1. Create a Hugging Face write token.
2. In Colab, open **Secrets**.
3. Add the token as `HF_TOKEN`.
4. Make sure notebook access to the secret is enabled.

The domain-adapted model is the final model used by the Streamlit app.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

repo_name = "chase1zhang/youtube-emotion-distilbert"
domain_repo_name = "chase1zhang/youtube-emotion-distilbert-domain-adapted"

trainer.model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)
print(f"Uploaded Stage 1 model to https://huggingface.co/{repo_name}")

domain_trainer.model.push_to_hub(domain_repo_name)
tokenizer.push_to_hub(domain_repo_name)
print(f"Uploaded domain-adapted model to https://huggingface.co/{domain_repo_name}")


## 17. Verify uploaded Hugging Face models

Run this after upload to confirm the model repositories contain valid model weights and can be loaded with `pipeline()`.


In [ ]:
uploaded_pipeline = pipeline("text-classification", model=repo_name, tokenizer=repo_name)
uploaded_domain_pipeline = pipeline("text-classification", model=domain_repo_name, tokenizer=domain_repo_name)

print("Stage 1 uploaded model:")
print(uploaded_pipeline(samples, truncation=True, return_token_type_ids=False))

print("Domain-adapted uploaded model:")
print(uploaded_domain_pipeline(samples, truncation=True, return_token_type_ids=False))


## 18. Final numbers to copy into the report

After the notebook finishes, copy these values into the project report and experimental results workbook:

- Stage 1 GoEmotions validation metrics
- Stage 1 GoEmotions test metrics
- Stage 2 YouTube-domain validation metrics
- Hugging Face model URLs

The Streamlit app default model should remain:

```python
chase1zhang/youtube-emotion-distilbert-domain-adapted
```
